# v2 · Notebook 02 — Treinar o nosso cross-encoder (ajuste fino)

Ideia trazida de **Nogueira & Cho (2019)**, **mMARCO (Bonifacio et al.)** e **ptt5-v2**: em vez de só usar um
reranqueador pronto, **ajustamos o nosso próprio** cross-encoder nas triplas de treino do mMARCO em português.

- **Ponto de partida:** `nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large`, o mesmo modelo base do
  `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (conferido no `config.json` dele). Mesma arquitetura, então a
  comparação com o modelo pronto isola **o efeito do treinamento**.
- **Tarefa:** cada tripla vira 2 pares rotulados: (consulta, positiva) → 1 e (consulta, negativa) → 0.
- **Perda:** entropia cruzada binária sobre o logit (LEARN.md, Parte 5.4).
- **Laço de treino escrito à mão** (PyTorch + `transformers`), sem `CrossEncoder.fit`: é a parte "implementar" do objetivo.

**Feito para o Colab gratuito (GPU T4):** salva um ponto de retomada no Drive a cada `RESUME_EVERY_MIN` minutos.
Se a sessão cair, reconecte, rode todas as células de novo, e o treino **continua de onde parou**.
Também salva *snapshots* em frações do treino para a **curva de aprendizado** do notebook 03.

Antes: Ambiente de execução → Alterar o tipo de ambiente → **GPU T4**.

**Tempo estimado** (não medido): ~45 min para 150 mil triplas (300 mil pares) numa T4, extrapolado dos 2,9 ms/par
de inferência da v1. A célula de treino imprime a velocidade real e o tempo restante a cada 100 passos.
Se a sessão do Colab gratuito costuma cair antes disso, reduza `N_TRIPLES_USE` ou simplesmente retome.

Se `from_pretrained` reclamar de `torch.load` / versão do torch ao carregar o modelo base (ele vem em `pytorch_model.bin`),
rode `!pip install -q -U torch` e reinicie o ambiente.

In [ ]:
# ==============================================================================
# Google Drive: tudo o que precisa sobreviver a uma desconexão fica aqui
# ==============================================================================
import os
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/AssignmentPLN_v2"
DATA = f"{BASE}/data"          # saída do notebook 01
CKPT = f"{BASE}/checkpoints"   # saída do notebook 02
RES  = f"{BASE}/results"       # saída do notebook 03
for d in (DATA, CKPT, RES):
    os.makedirs(d, exist_ok=True)

SEED = 42

In [ ]:
import json, math, time, random, shutil
import numpy as np
import pandas as pd
import torch
from torch.nn import BCEWithLogitsLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import transformers

assert torch.cuda.is_available(), "Ative a GPU: Ambiente de execução -> Alterar o tipo de ambiente -> T4"
DEVICE = "cuda"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__)

# ---------------- Configuração ----------------
BASE_MODEL       = "nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large"
N_TRIPLES_USE    = 150_000   # triplas usadas (≤ N_TRAIN_TRIPLES do notebook 01). 150k = 300k pares
BATCH_TRIPLES    = 16        # 16 triplas = 32 pares por passo
LR               = 2e-5
WARMUP_FRAC      = 0.1
WEIGHT_DECAY     = 0.01
MAX_LEN          = 256       # passagens do MS MARCO são curtas; 256 tokens cobre quase todas
SNAPSHOT_FRACS   = [0.05, 0.1, 0.25, 0.5, 1.0]   # pontos da curva de aprendizado
RESUME_EVERY_MIN = 15
LOG_EVERY        = 100

RUN_NAME = f"mminilm_bce_{N_TRIPLES_USE//1000}k_lr{LR}_bs{2*BATCH_TRIPLES}"
RUN_DIR  = f"{CKPT}/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
df = pd.read_parquet(f"{DATA}/train_triples.parquet")
tr = df[df.split == "train"].reset_index(drop=True)
va = df[df.split == "val"].reset_index(drop=True)
assert len(tr) >= N_TRIPLES_USE, f"só há {len(tr)} triplas preparadas"
tr = tr.iloc[:N_TRIPLES_USE]   # o notebook 01 já embaralhou

order = np.random.default_rng(SEED).permutation(len(tr))   # ordem fixa -> retomada determinística
TOTAL_STEPS = len(tr) // BATCH_TRIPLES
SNAPSHOT_STEPS = sorted({max(1, int(round(f * TOTAL_STEPS))) for f in SNAPSHOT_FRACS})
print(f"treino: {len(tr):,} triplas = {2*len(tr):,} pares | {TOTAL_STEPS:,} passos | snapshots em {SNAPSHOT_STEPS}")
print(f"validação: {len(va):,} triplas")

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
# num_labels=1 -> cabeça nova, aleatória, que produz 1 logit (o aviso "newly initialized" é esperado)
# .float(): o checkpoint base está salvo em fp16 (pytorch_model.bin = 236 MB ≈ 118M × 2 bytes) e versões novas
# do transformers carregam no dtype salvo. Os pesos-mestre PRECISAM ser fp32: o autocast faz as contas em fp16
# e o GradScaler só funciona com gradientes fp32 (senão: "Attempting to unscale FP16 gradients").
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1).float().to(DEVICE)
_dtypes = {p.dtype for p in model.parameters()}
assert _dtypes == {torch.float32}, f"parâmetros não estão em fp32: {_dtypes}"
print("parâmetros:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M", "| dtype:", _dtypes)

no_decay = ("bias", "LayerNorm.weight")
params = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(params, lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * TOTAL_STEPS), TOTAL_STEPS)
scaler = torch.amp.GradScaler("cuda")       # fp16 misto: ~2x mais rápido na T4
loss_fn = BCEWithLogitsLoss()

def encode(queries, passages):
    return tok(queries, passages, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)

def make_batch(rows):
    n = len(rows)
    q = list(rows["query"]) * 2
    p = list(rows["pos_text"]) + list(rows["neg_text"])
    y = torch.tensor([1.0] * n + [0.0] * n, device=DEVICE)
    return encode(q, p), y

In [ ]:
@torch.no_grad()
def validar(bs=64):
    """Acurácia por par (score(pos) > score(neg)) e perda média na validação."""
    model.eval()
    acertos, perdas = 0, []
    for i in range(0, len(va), bs):
        rows = va.iloc[i:i+bs]
        enc, y = make_batch(rows)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits.squeeze(-1).float()
        perdas.append(loss_fn(logits, y).item())
        n = len(rows)
        acertos += (logits[:n] > logits[n:]).sum().item()
    model.train()
    return acertos / len(va), float(np.mean(perdas))

RESUME = f"{RUN_DIR}/resume.pt"
CURVE  = f"{RUN_DIR}/curve.json"

def salvar_retomada(step):
    tmp = "/content/resume_tmp.pt"
    torch.save({"step": step, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state()}, tmp)
    shutil.copy(tmp, RESUME + ".part")      # grava no Drive sob outro nome e só então substitui:
    os.replace(RESUME + ".part", RESUME)    # uma queda no meio da cópia não corrompe a retomada anterior
    print(f"  [retomada salva no passo {step}]")

def salvar_snapshot(step):
    """Snapshot leve (pesos em fp16, ~240 MB) para a curva de aprendizado."""
    acc, vloss = validar()
    sd = {k: v.detach().half().cpu() for k, v in model.state_dict().items()}
    torch.save(sd, f"{RUN_DIR}/snap_step{step}.pt")
    curve = json.load(open(CURVE)) if os.path.exists(CURVE) else []
    curve = [c for c in curve if c["step"] != step] + [{
        "step": step, "pairs_seen": step * 2 * BATCH_TRIPLES, "val_pair_acc": acc, "val_loss": vloss,
        "file": f"snap_step{step}.pt"}]
    json.dump(sorted(curve, key=lambda c: c["step"]), open(CURVE, "w"), indent=2)
    print(f"  [snapshot passo {step}: acurácia de par na validação = {acc:.4f}, perda = {vloss:.4f}]")

start_step = 0
if os.path.exists(RESUME):
    st = torch.load(RESUME, map_location=DEVICE, weights_only=False)
    model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
    scheduler.load_state_dict(st["scheduler"]); scaler.load_state_dict(st["scaler"])
    # map_location=DEVICE também levou os estados de RNG para a GPU; set_rng_state exige ByteTensor na CPU
    torch.set_rng_state(st["torch_rng"].cpu()); torch.cuda.set_rng_state(st["cuda_rng"].cpu())
    start_step = st["step"]
    print(f"Retomando do passo {start_step}/{TOTAL_STEPS}")
else:
    acc0, l0 = validar()
    print(f"Antes do treino: acurácia de par = {acc0:.4f} (≈0,5 = acaso), perda = {l0:.4f}")

In [ ]:
model.train()
t_start, t_last_resume, run_loss = time.time(), time.time(), []

for step in range(start_step + 1, TOTAL_STEPS + 1):
    idx = order[(step - 1) * BATCH_TRIPLES: step * BATCH_TRIPLES]
    enc, y = make_batch(tr.iloc[idx])

    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(**enc).logits.squeeze(-1)
    loss = loss_fn(logits.float(), y)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer); scaler.update(); scheduler.step()
    run_loss.append(loss.item())

    if step % LOG_EVERY == 0:
        el = time.time() - t_start
        vel = (step - start_step) / el
        eta = (TOTAL_STEPS - step) / vel / 60
        print(f"passo {step:>6}/{TOTAL_STEPS} | perda {np.mean(run_loss):.4f} | lr {scheduler.get_last_lr()[0]:.2e} "
              f"| {vel*2*BATCH_TRIPLES:,.0f} pares/s | faltam ~{eta:.0f} min")
        run_loss = []

    if step in SNAPSHOT_STEPS:
        salvar_snapshot(step)
    if time.time() - t_last_resume > RESUME_EVERY_MIN * 60 and step < TOTAL_STEPS:
        salvar_retomada(step); t_last_resume = time.time()

print(f"Treino concluído em {(time.time()-t_start)/60:.1f} min (nesta sessão).")

In [ ]:
# Modelo final completo (fp32), carregável com from_pretrained no notebook 03
FINAL = f"{RUN_DIR}/final"
model.save_pretrained(FINAL); tok.save_pretrained(FINAL)
json.dump({"base_model": BASE_MODEL, "N_TRIPLES_USE": N_TRIPLES_USE, "BATCH_TRIPLES": BATCH_TRIPLES, "LR": LR,
           "WARMUP_FRAC": WARMUP_FRAC, "WEIGHT_DECAY": WEIGHT_DECAY, "MAX_LEN": MAX_LEN, "TOTAL_STEPS": TOTAL_STEPS,
           "seed": SEED, "torch": torch.__version__, "transformers": transformers.__version__,
           "gpu": torch.cuda.get_device_name(0)},
          open(f"{RUN_DIR}/train_config.json", "w"), indent=2)
if os.path.exists(RESUME):
    os.remove(RESUME)   # ~1,4 GB; não é mais necessário
print("Salvo em", FINAL)
print(json.dumps(json.load(open(CURVE)), indent=2))